# Modul 05: Analisis Korelasi dan Seleksi Fitur
**Mata Kuliah:** Statistika Komputasi  
**Dosen Pengampu:** Dr. Ridwan Ilyas, S.Kom., M.T.  
**Program Studi:** Teknik Informatika, Universitas Jenderal Achmad Yani (UNJANI) 2026  
**Lisensi:** Open Source (MIT)

---

## 📖 1. Analisis Korelasi dan Seleksi Fitur

Analisis korelasi mengukur derajat dan arah hubungan linier maupun monotonik antara dua atau lebih variabel acak:
1. **Koefisien Korelasi Pearson ($r$)**:
   - Digunakan untuk data berpasangan pada skala interval/rasio dengan asumsi bivariat normal.
   - Formula: $r = 
rac{\sum (X_i - ar{X})(Y_i - ar{Y})}{\sqrt{\sum (X_i - ar{X})^2 \sum (Y_i - ar{Y})^2}}$, bernilai $-1.0 \le r \le +1.0$.
2. **Koefisien Korelasi Spearman Rank ($
ho$)**:
   - Metode non-parametrik berbasis urutan peringkat (rank) untuk data ordinal atau hubungan monotonik non-linier.
3. **Korelasi Parsial**:
   - Mengukur kekuatan hubungan murni antara variabel $X$ dan $Y$ setelah **mengontrol/menghilangkan pengaruh variabel perancu $Z$** (*confounding variable*).
4. **Seleksi Fitur (*Feature Selection*)**:
   - Menyaring fitur prediktor yang memiliki korelasi tinggi terhadap target ($|r_{X, Y}| > 0.3$), sekaligus membuang fitur yang saling berkorelasi sangat kuat satu sama lain ($|r_{X_i, X_j}| > 0.85$) untuk mencegah redundansi informasi.


## 📊 2. Diagram Ilustrasi Konsep

![Ilustrasi Analisis Korelasi & Seleksi Fitur](images/img_05_correlation_feature_selection.png)

```
        +-------------------------------------------------------------+
        |                 SPEKTRUM KOEFISIEN KORELASI (r)             |
        +-------------------------------------------------------------+
        |  -1.0 (Korelasi Negatif Sempurna) <--- Garis Menurun        |
        |   0.0 (Tidak Ada Hubungan Linier) <--- Sebaran Acak         |
        |  +1.0 (Korelasi Positif Sempurna) <--- Garis Menaik         |
        +-------------------------------------------------------------+
```


## 🔬 3. Studi Kasus & Penjelasan Langkah Komputasi

Studi kasus menganalisis interaksi pengguna pada aplikasi web (`03_user_engagement_correlation.csv`) yang mencakup waktu sesi aktif, jumlah klik fitur, dan total pengeluaran belanja.

**Tahapan Komputasi:**
1. Menghitung matriks korelasi Pearson dan Spearman Rank.
2. Memvisualisasikan peta panas (*Correlation Heatmap*) dengan anotasi nilai koefisien.
3. Menghitung korelasi parsial antara Waktu Sesi Aktif dan Total Belanja setelah mengontrol variabel Jumlah Klik.


In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import pingouin as pg
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
df_eng = pd.read_csv("../datasets/03_user_engagement_correlation.csv")
print("Dataset interaksi pengguna dimuat:", df_eng.shape)
display(df_eng.head())


## 💻 4. Eksekusi Komputasi Python & Matriks Korelasi


In [ ]:
# 1. Perhitungan Matriks Korelasi Pearson
corr_pearson = df_eng.drop(columns=['user_id']).corr(method='pearson')
print("=== Matriks Korelasi Pearson ===")
display(corr_pearson.round(3))

# 2. Korelasi Parsial (Mengontrol Jumlah Klik)
partial_res = pg.partial_corr(data=df_eng, x='session_duration_mins', y='total_spend_k', covar='clicks_count')
print("
=== Hasil Uji Korelasi Parsial ===")
display(partial_res[['n', 'r', 'p-val']].round(4))


In [ ]:
# 3. Visualisasi Heatmap dan Scatter Korelasi
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Heatmap Korelasi
sns.heatmap(corr_pearson, annot=True, cmap="coolwarm", vmin=-1, vmax=1, fmt=".2f", ax=axes[0])
axes[0].set_title('Matriks Korelasi Fitur Interaksi', fontweight='bold')

# Scatter Plot Regresi
sns.regplot(data=df_eng, x='session_duration_mins', y='total_spend_k', 
            scatter_kws={'color': '#1A365D', 'alpha': 0.7}, line_kws={'color': '#EA580C', 'linewidth': 2.5}, ax=axes[1])
axes[1].set_title('Hubungan Linier: Durasi Sesi vs. Total Belanja (r = +0.82)', fontweight='bold')
axes[1].set_xlabel('Durasi Sesi Aktif (Menit)')
axes[1].set_ylabel('Total Belanja (K IDR)')

plt.tight_layout()
plt.show()


## 📝 5. Kesimpulan Analisis & Data Storytelling

### ❓ Pertanyaan Refleksi & Konsep
* **Apakah korelasi membuktikan adanya hubungan sebab-akibat (kausalitas)?** Tidak (*Correlation does not imply Causation*). Korelasi hanya membuktikan ko-variasi statistik. Kausalitas memerlukan desain eksperimen terkontrol (A/B testing) atau analisis kausal formal.

### 🔍 Temuan Utama Data (Key Findings)
* **Durasi Sesi Aktif** berkorelasi positif sangat kuat dengan **Total Belanja** ($r = +0.82$, $p < 0.001$).
* Setelah mengontrol variabel perantara *Jumlah Klik*, hubungan murni parsial tetap signifikan positif ($r_{partial} = +0.58$), membuktikan bahwa durasi aplikasi secara langsung mendorong belanja pengguna.

### 💡 Rekomendasi & Langkah Lanjutan
* Fitur Durasi Sesi dan Jumlah Fitur yang Diakses diprioritaskan masuk ke dalam model regresi prediktif pada Modul 06.
